
# Unified Job Postings Builder (Two Folders)

This notebook builds a single **`public.job_postings_unified`** table from **two** local dumps:

- `240826_panel_data_with_dg` (has `duplicate_group`)
- `250826_panel_data_01012024_30062025` (may not have `duplicate_group`)

It also:
- Creates a text normalization function and a **generated** `content_norm` column
- Indexes **both** `content_norm` and **`content_clean`** for fast `ILIKE`/regex
- Adds BRIN/BTREE indexes for your other access paths
- Streams `.json.gzip` NDJSON files and ingests with `ON CONFLICT (uid) DO NOTHING`
- Runs quick sanity checks and sample queries

> Run cells **top-to-bottom**. Ensure your `config.env` has DB credentials.


In [ ]:
# === UNIFIED LOADER w/ SURROGATE UID (two folders) ===
# Fix: pass duplicate_group as TEXT (str) so PG will cast to UUID
# Also: more forgiving UID synthesis (fallbacks if company_id missing)

import os, re, gzip, json, unicodedata, hashlib, uuid
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

# -----------------------------
# DB connect
# -----------------------------
load_dotenv("config.env")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")

conn = psycopg2.connect(
    dbname=DB_NAME, user=DB_USER, password=DB_PASS, host=DB_HOST, port=DB_PORT
)
conn.autocommit = False
print(f"✅ Connected to Postgres as {DB_USER} on {DB_HOST}")

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS btree_gin;")
    cur.execute("""
    CREATE TABLE IF NOT EXISTS public.job_postings_unified (
        uid                 TEXT PRIMARY KEY,
        title               TEXT,
        company_id          TEXT,
        company_name        TEXT,
        company_is_recruiter BOOLEAN,
        company_size        TEXT,
        cantons             TEXT,
        x28_industries      TEXT,
        x28_occupations     TEXT,
        location_raw        TEXT,
        url                 TEXT,
        content_clean       TEXT,
        content_norm        TEXT,
        duplicate_group     UUID NULL,
        tst_created         TIMESTAMPTZ,
        tst_deleted         TIMESTAMPTZ,
        source_folder       TEXT,
        source_file         TEXT
    );
    """)
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_brin_created
                   ON public.job_postings_unified USING BRIN (tst_created);""")
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_company_id
                   ON public.job_postings_unified (company_id);""")
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_company_name
                   ON public.job_postings_unified (company_name);""")
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_dupgroup
                   ON public.job_postings_unified (duplicate_group);""")
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_content_clean_trgm
                   ON public.job_postings_unified USING GIN (content_clean gin_trgm_ops);""")
    cur.execute("""CREATE INDEX IF NOT EXISTS job_postings_unified_content_norm_trgm
                   ON public.job_postings_unified USING GIN (content_norm gin_trgm_ops);""")
print("✅ Schema & indexes ensured.")

# -----------------------------
# Folders & files
# -----------------------------
BASE = Path("/Users/bradyallardice/Desktop/PhD/Projects/KurerAllardice2024/10CompaniesPOCAI/")
FOLDERS = [
    BASE / "240826_panel_data_with_dg",
    BASE / "250826_panel_data_01012024_30062025",
]

def list_json_gz(folder: Path):
    return list(folder.rglob("*.json.gzip"))

files = []
for fld in FOLDERS:
    files.extend(list_json_gz(fld))
files = sorted(set(p.resolve() for p in files))
print(f"📦 Files found (deduped): {len(files):,}")
if not files:
    raise SystemExit("No .json.gzip files found—check folder paths.")

# -----------------------------
# Helpers
# -----------------------------
def normalize_text_for_matching(text):
    if text is None:
        return None
    t = unicodedata.normalize("NFD", str(text)).encode("ascii","ignore").decode("ascii")
    t = t.lower()
    t = re.sub(r"\s+", " ", t).strip()
    return t

def ndjson_stream(path: Path):
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

def first_non_null(d, *keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

def synthesize_uid_from_fields(*parts):
    # Deterministic surrogate uid from any available parts
    key = "|".join("" if p is None else str(p) for p in parts)
    return "surr_" + hashlib.sha1(key.encode("utf-8")).hexdigest()

def choose_surrogate_uid(obj):
    company_id   = first_non_null(obj, "company_id")
    company_name = first_non_null(obj, "company_name", "companyName", "company")
    url          = first_non_null(obj, "url")
    title        = first_non_null(obj, "title")
    tst_created  = first_non_null(obj, "tst_created", "created_at", "posted_at", "createdAt")

    # Try combos in order of stability
    combos = [
        (company_id, url, tst_created),
        (company_id, url, title, tst_created),
        (company_name, url, tst_created),
        (company_name, title, url, tst_created),
    ]
    for tup in combos:
        if all(x not in (None, "") for x in tup):
            return synthesize_uid_from_fields(*tup)
    return None

# Columns / SQL
COLS = [
    "uid","title","company_id","company_name","company_is_recruiter","company_size",
    "cantons","x28_industries","x28_occupations","location_raw","url",
    "content_clean","content_norm","duplicate_group","tst_created","tst_deleted",
    "source_folder","source_file",
]
insert_sql = f"""
INSERT INTO public.job_postings_unified ({", ".join(COLS)})
VALUES %s
ON CONFLICT (uid) DO NOTHING;
"""

def build_row(obj, source_folder, source_file):
    # Preferred native uid
    native_uid   = first_non_null(obj, "uid", "id")
    title        = first_non_null(obj, "title")
    company_name = first_non_null(obj, "company_name", "companyName", "company")
    company_id   = first_non_null(obj, "company_id")
    url          = first_non_null(obj, "url")
    tst_created  = first_non_null(obj, "tst_created", "created_at", "posted_at", "createdAt")

    uid = str(native_uid) if native_uid else choose_surrogate_uid(obj)
    if not uid:
        return None  # cannot build stable key

    # Require critical downstream fields
    if not (title and company_name and tst_created and url):
        return None

    content_clean = obj.get("content_clean")
    content_norm  = normalize_text_for_matching(content_clean)

    # >>> KEY FIX: cast UUID to string or None <<<
    raw_dup = obj.get("duplicate_group")
    dup_str = None
    if raw_dup not in (None, ""):
        try:
            dup_str = str(uuid.UUID(str(raw_dup)))
        except Exception:
            dup_str = None  # invalid uuid -> store NULL

    return (
        uid,
        title,
        company_id,
        company_name,
        obj.get("company_is_recruiter"),
        obj.get("company_size"),
        obj.get("cantons"),
        obj.get("x28_industries"),
        obj.get("x28_occupations"),
        obj.get("location_raw"),
        url,
        content_clean,
        content_norm,
        dup_str,              # <-- text; PG will cast to UUID
        tst_created,
        obj.get("tst_deleted"),
        source_folder,
        source_file,
    )

def insert_batch(cur, rows):
    if not rows:
        return 0
    execute_values(cur, insert_sql, rows, page_size=min(len(rows), 5000))
    return len(rows)

# -----------------------------
# Pilot insert (first file, up to 10k lines)
# -----------------------------
pilot = files[0]
print(f"🔎 Pilot file: {pilot}")
reasons = Counter()
sample_keys = None
pilot_rows, kept, skipped = [], 0, 0

for i, obj in enumerate(ndjson_stream(pilot)):
    if sample_keys is None:
        sample_keys = sorted(obj.keys())
    row = build_row(obj, source_folder=pilot.parent.name, source_file=pilot.name)
    if row is None:
        # audit why (coarse)
        if not first_non_null(obj, "uid", "id") and choose_surrogate_uid(obj) is None:
            reasons["no_uid_or_surrogate_inputs"] += 1
        else:
            # missing critical fields
            if not first_non_null(obj, "title"):
                reasons["no_title"] += 1
            if not first_non_null(obj, "company_name", "companyName", "company"):
                reasons["no_company_name"] += 1
            if not first_non_null(obj, "tst_created", "created_at", "posted_at", "createdAt"):
                reasons["no_tst_created"] += 1
            if not first_non_null(obj, "url"):
                reasons["no_url"] += 1
        skipped += 1
    else:
        pilot_rows.append(row)
        kept += 1
    if i >= 9999:
        break

print("Pilot keys:", sample_keys)
print(f"Pilot kept={kept:,} skipped={skipped:,} (first 10k)")
print("Skip reasons:", dict(reasons))

if kept > 0:
    with conn.cursor() as cur:
        cur.execute("SET LOCAL synchronous_commit = off;")
        n = insert_batch(cur, pilot_rows[:2000])
    conn.commit()
    print(f"🧪 Pilot insert OK: {n} rows inserted.")
else:
    print("🚫 Pilot produced 0 rows; fix inputs and rerun.")

# -----------------------------
# Full load if pilot ok
# -----------------------------
if 'n' in locals() and n > 0:
    total_scanned = total_inserted = 0
    for fld in FOLDERS:
        scanned = inserted_here = 0
        these = sorted(set(p.resolve() for p in list_json_gz(fld)))
        for path in these:
            batch = []
            for obj in ndjson_stream(path):
                scanned += 1
                total_scanned += 1
                row = build_row(obj, source_folder=fld.name, source_file=path.name)
                if row is None:
                    continue
                batch.append(row)
                if len(batch) >= 5000:
                    with conn.cursor() as cur:
                        cur.execute("SET LOCAL synchronous_commit = off;")
                        inserted_here += insert_batch(cur, batch)
                    batch.clear()
            if batch:
                with conn.cursor() as cur:
                    cur.execute("SET LOCAL synchronous_commit = off;")
                    inserted_here += insert_batch(cur, batch)
                batch.clear()
        total_inserted += inserted_here
        print(f"[{fld.name}] scanned≈{scanned:,} inserted={inserted_here:,}")

    conn.commit()
    print("✅ Load complete.")
    with conn.cursor() as cur:
        cur.execute("ANALYZE public.job_postings_unified;")
    print("✅ Table analyzed.")

    # -----------------------------
    # QA
    # -----------------------------
    def q(sql_text, params=None):
        return pd.read_sql_query(sql_text, conn, params=params)

    print("\nRow counts by source:")
    print(q("""SELECT source_folder, COUNT(*) AS n
               FROM public.job_postings_unified
               GROUP BY source_folder ORDER BY source_folder;"""))

    print("\nDate ranges by source:")
    print(q("""SELECT source_folder,
                      MIN(tst_created) AS min_created,
                      MAX(tst_created) AS max_created
               FROM public.job_postings_unified
               GROUP BY source_folder ORDER BY source_folder;"""))

    print("\nDuplicate group coverage by source:")
    print(q("""SELECT source_folder,
                      COUNT(*) AS total,
                      COUNT(duplicate_group) AS with_dupgroup,
                      ROUND(100.0 * COUNT(duplicate_group)::numeric / NULLIF(COUNT(*),0), 2) AS pct_with_dupgroup
               FROM public.job_postings_unified
               GROUP BY source_folder ORDER BY source_folder;"""))

    print("\nSample 5 rows:")
    print(q("""SELECT uid, company_name, title, tst_created, LEFT(content_clean, 120) AS snippet
               FROM public.job_postings_unified
               ORDER BY tst_created DESC
               LIMIT 5;"""))

    print("\nSearch smoke test (ILIKE + regex):")
    print(q(r"""SELECT uid, company_name, title, tst_created
                FROM public.job_postings_unified
                WHERE content_clean ILIKE '%machine learning%'
                   OR content_norm ~* '\yai\y'
                ORDER BY tst_created DESC
                LIMIT 10;"""))
else:
    print("\n🛑 Stopping before full load because pilot insert didn’t succeed.")


✅ Connected to Postgres as ballardice on localhost
✅ Schema & indexes ensured.
📦 Files found (deduped): 191
🔎 Pilot file: /Users/bradyallardice/Desktop/PhD/Projects/KurerAllardice2024/10CompaniesPOCAI/240826_panel_data_with_dg/data_000000000000.json.gzip
Pilot keys: ['cantons', 'company_id', 'company_is_recruiter', 'company_name', 'company_size', 'content_clean', 'duplicate_group', 'location_raw', 'title', 'tst_created', 'tst_deleted', 'url', 'x28_industries', 'x28_occupations']
Pilot kept=10,000 skipped=0 (first 10k)
Skip reasons: {}


In [11]:

DDL_SQL = r"""
-- 1) Extensions (safe if already exist)
CREATE EXTENSION IF NOT EXISTS unaccent;
CREATE EXTENSION IF NOT EXISTS pg_trgm;

-- 2) Normalization function (IMMUTABLE for generated cols & indexing)
CREATE OR REPLACE FUNCTION public.content_normalize(t text)
RETURNS text
LANGUAGE sql
IMMUTABLE
AS $$
  SELECT
    CASE
      WHEN t IS NULL THEN NULL
      ELSE
        trim(
          regexp_replace(
            regexp_replace(
              lower(unaccent(t)),
              '[^[:alnum:]\s]+', ' ', 'g'  -- strip punctuation/specials
            ),
            '\s+', ' ', 'g'                -- collapse whitespace
          )
        )
    END
$$;

-- 3) Canonical table with GENERATED content_norm
CREATE TABLE IF NOT EXISTS public.job_postings_unified (
  uid              uuid PRIMARY KEY,
  title            text,
  company_id       text,
  company_name     text,
  company_is_recruiter boolean,
  company_size     text,
  cantons          text[],
  x28_industries   text[],
  x28_occupations  text[],
  location_raw     text,
  url              text,
  content_clean    text,
  content_norm     text GENERATED ALWAYS AS (public.content_normalize(content_clean)) STORED,
  duplicate_group  uuid,
  tst_created      timestamptz,
  tst_deleted      timestamptz,
  source_folder    text NOT NULL,
  source_file      text NOT NULL
);
"""

with conn.cursor() as cur:
    cur.execute(DDL_SQL)

print("✅ Extensions, function, and table ensured.")


✅ Extensions, function, and table ensured.


In [12]:

from typing import Iterable, Dict, Any, List

ARRAY_FIELDS = {"cantons", "x28_industries", "x28_occupations"}

def to_uuid_or_none(v):
    if v in (None, "", "null"): return None
    try: return uuid.UUID(str(v))
    except Exception: return None

def to_array(v):
    if v is None: return None
    if isinstance(v, list): return [str(x) if x is not None else None for x in v]
    s = str(v).strip()
    if not s: return None
    # Try JSON parse
    try:
        j = json.loads(s)
        if isinstance(j, list):
            return [str(x) if x is not None else None for x in j]
    except Exception:
        pass
    # Fallback on split
    parts = [p.strip() for p in s.replace(";", ",").split(",")]
    return [p for p in parts if p]

def coerce_row(raw: Dict[str, Any], source_folder: str, source_file: str) -> Dict[str, Any]:
    def g(*names, default=None):
        for n in names:
            if n in raw: return raw[n]
        return default

    return {
        "uid":                 to_uuid_or_none(g("uid")),
        "title":               g("title"),
        "company_id":          g("company_id"),
        "company_name":        g("company_name"),
        "company_is_recruiter":g("company_is_recruiter"),
        "company_size":        g("company_size"),
        "cantons":             to_array(g("cantons")),
        "x28_industries":      to_array(g("x28_industries")),
        "x28_occupations":     to_array(g("x28_occupations")),
        "location_raw":        g("location_raw"),
        "url":                 g("url"),
        "content_clean":       g("content_clean"),
        "duplicate_group":     to_uuid_or_none(g("duplicate_group")),
        "tst_created":         g("tst_created"),
        "tst_deleted":         g("tst_deleted"),
        "source_folder":       source_folder,
        "source_file":         source_file,
    }

def stream_ndjson_gz(path: Path) -> Iterable[Dict[str, Any]]:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def bulk_insert(conn, rows: List[Dict[str, Any]]):
    cols = [
        "uid","title","company_id","company_name","company_is_recruiter","company_size",
        "cantons","x28_industries","x28_occupations","location_raw","url",
        "content_clean","duplicate_group","tst_created","tst_deleted","source_folder","source_file"
    ]
    tpl = "(" + ",".join(["%s"]*len(cols)) + ")"
    values = [[r.get(c) for c in cols] for r in rows]
    with conn.cursor() as cur:
        execute_values(cur, f"""
            INSERT INTO public.job_postings_unified ({",".join(cols)})
            VALUES %s
            ON CONFLICT (uid) DO NOTHING
        """, values, template=tpl, page_size=10_000)

def load_folder(conn, folder: str, batch_size: int = BATCH_SIZE):
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"⚠️  Folder not found: {folder_path.resolve()} (skipping)")
        return

    gz_files = sorted(folder_path.glob("*.json.gzip")) or sorted(folder_path.glob("*.gzip")) or sorted(folder_path.glob("*.gz"))
    if not gz_files:
        print(f"⚠️  No .json.gzip/.gzip/.gz files in {folder_path} (skipping)")
        return

    total = 0; inserted = 0; batch = []
    for gz in gz_files:
        for rec in stream_ndjson_gz(gz):
            total += 1
            row = coerce_row(rec, folder_path.name, gz.name)
            if row["uid"] is None:
                continue
            batch.append(row)
            if len(batch) >= batch_size:
                bulk_insert(conn, batch); inserted += len(batch); batch.clear()
    if batch:
        bulk_insert(conn, batch); inserted += len(batch)

    print(f"[{folder_path.name}] scanned={total:,} inserted={inserted:,}")


In [13]:

for folder in FOLDERS:
    load_folder(conn, folder)

with conn.cursor() as cur:
    cur.execute("ANALYZE public.job_postings_unified;")

print("✅ Load complete and analyzed.")


[240826_panel_data_with_dg] scanned=6,006,115 inserted=0
[250826_panel_data_01012024_30062025] scanned=955,113 inserted=0
✅ Load complete and analyzed.


In [14]:

INDEX_SQL = r"""
-- Trigram indexes for ILIKE / regex
CREATE INDEX IF NOT EXISTS job_unified_content_norm_trgm
  ON public.job_postings_unified USING gin (content_norm gin_trgm_ops);

CREATE INDEX IF NOT EXISTS job_unified_content_clean_trgm
  ON public.job_postings_unified USING gin (content_clean gin_trgm_ops);

-- Date / range scans
CREATE INDEX IF NOT EXISTS job_unified_tst_created_brin
  ON public.job_postings_unified USING brin (tst_created);

-- Common filters / joins
CREATE INDEX IF NOT EXISTS job_unified_company_id_btree
  ON public.job_postings_unified (company_id);

CREATE INDEX IF NOT EXISTS job_unified_company_name_btree
  ON public.job_postings_unified (company_name);

CREATE INDEX IF NOT EXISTS job_unified_dupgroup_btree
  ON public.job_postings_unified (duplicate_group);

-- Ordered retrieval per company
CREATE INDEX IF NOT EXISTS job_unified_company_name_created
  ON public.job_postings_unified (company_name, tst_created DESC);
"""

with conn.cursor() as cur:
    cur.execute(INDEX_SQL)
    cur.execute("ANALYZE public.job_postings_unified;")

print("✅ Indexes created and table analyzed.")


✅ Indexes created and table analyzed.


In [15]:
import os, psycopg2, pandas as pd
from dotenv import load_dotenv

load_dotenv("config.env")
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
)
conn.autocommit = True
cur = conn.cursor()

def q(sql, params=None):
    cur.execute(sql, params or [])
    return cur.fetchall()

print("DB:", q("SELECT current_database()")[0][0])
print("Schema search path:", q("SHOW search_path")[0][0])

# Does the table exist where we expect?
exists = q("""
SELECT to_regclass('public.job_postings_unified')
""")[0][0]
print("Table present?:", bool(exists))

# Row count + date range
if exists:
    print("Counts/min/max:", q("""
        SELECT COUNT(*), MIN(tst_created), MAX(tst_created)
        FROM public.job_postings_unified
    """)[0])

# Peek columns so we know we’re hitting the right object
if exists:
    cols = q("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema='public' AND table_name='job_postings_unified'
        ORDER BY ordinal_position
    """)
    print("Columns:", [c[0] for c in cols])


DB: kurerallardice2024
Schema search path: "$user", public
Table present?: True
Counts/min/max: (0, None, None)
Columns: ['uid', 'title', 'company_id', 'company_name', 'company_is_recruiter', 'company_size', 'cantons', 'x28_industries', 'x28_occupations', 'location_raw', 'url', 'content_clean', 'content_norm', 'duplicate_group', 'tst_created', 'tst_deleted', 'source_folder', 'source_file']


In [16]:

import pandas as pd

def qdf(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("Row counts by source:")
display(qdf("""
SELECT source_folder, COUNT(*) AS n
FROM public.job_postings_unified
GROUP BY 1 ORDER BY 1
"""))

print("Date ranges by source:")
display(qdf("""
SELECT source_folder, MIN(tst_created) AS min_created, MAX(tst_created) AS max_created
FROM public.job_postings_unified
GROUP BY 1 ORDER BY 1
"""))

print("Duplicate group coverage by source:")
display(qdf("""
SELECT source_folder,
       COUNT(*) AS total,
       COUNT(duplicate_group) AS with_dupgroup,
       ROUND(100.0*COUNT(duplicate_group)/NULLIF(COUNT(*),0), 2) AS pct_with_dupgroup
FROM public.job_postings_unified
GROUP BY 1 ORDER BY 1
"""))


Row counts by source:


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,source_folder,n


Date ranges by source:


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,source_folder,min_created,max_created


Duplicate group coverage by source:


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,source_folder,total,with_dupgroup,pct_with_dupgroup


In [17]:

# Tune these to your needs:
SAMPLE_COMPANY = 'PPCmetrics AG'
KEYWORD_PHRASE = 'machine learning'  # example

print("Sample: latest jobs for a company")
display(qdf("""
SELECT uid, company_name, title, tst_created
FROM public.job_postings_unified
WHERE company_name = %(c)s
ORDER BY tst_created DESC
LIMIT 20
""", {"c": SAMPLE_COMPANY}))

print("Sample: ILIKE on *content_norm* (accelerated by trigram)")
display(qdf("""
EXPLAIN ANALYZE
SELECT uid, company_name, title, tst_created
FROM public.job_postings_unified
WHERE content_norm ILIKE %(p)s
ORDER BY tst_created DESC
LIMIT 20
""", {"p": f"%{KEYWORD_PHRASE}%" }))

print("Sample: ILIKE on *content_clean* (also accelerated by trigram)")
display(qdf("""
EXPLAIN ANALYZE
SELECT uid, company_name, title, tst_created
FROM public.job_postings_unified
WHERE content_clean ILIKE %(p)s
ORDER BY tst_created DESC
LIMIT 20
""", {"p": f"%{KEYWORD_PHRASE}%" }))


Sample: latest jobs for a company


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,uid,company_name,title,tst_created


Sample: ILIKE on *content_norm* (accelerated by trigram)


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,QUERY PLAN
0,Limit (cost=0.01..0.02 rows=1 width=88) (actu...
1,-> Sort (cost=0.01..0.02 rows=1 width=88) ...
2,Sort Key: tst_created DESC
3,Sort Method: quicksort Memory: 25kB
4,-> Seq Scan on job_postings_unified ...
5,Filter: (content_norm ~~* '%mach...
6,Planning Time: 36.375 ms
7,Execution Time: 0.045 ms


Sample: ILIKE on *content_clean* (also accelerated by trigram)


/var/folders/3p/bkhpj5cj7p77ghh4hx0hz_cw0000gn/T/ipykernel_8575/245249445.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,QUERY PLAN
0,Limit (cost=0.01..0.02 rows=1 width=88) (actu...
1,-> Sort (cost=0.01..0.02 rows=1 width=88) ...
2,Sort Key: tst_created DESC
3,Sort Method: quicksort Memory: 25kB
4,-> Seq Scan on job_postings_unified ...
5,Filter: (content_clean ~~* '%mac...
6,Planning Time: 0.784 ms
7,Execution Time: 0.032 ms


In [1]:
# ============================================================================
# FIXED IMPLEMENTATION: Staging Table Approach
# ============================================================================

import os, re, gzip, json, unicodedata, hashlib, uuid
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
import pandas as pd

print("🔧 Starting fixed implementation with staging table approach...")

# Connect to database
load_dotenv("config.env")
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)
conn.autocommit = False

# Schema and table names
SCHEMA = "public"
TABLE_FINAL = "job_postings_unified"
TABLE_STAGE = "job_postings_unified_staging"

print("🗂️ Creating staging table with all TEXT columns...")

# Create staging table with all TEXT columns to avoid type casting issues during COPY
with conn.cursor() as cur:
    cur.execute(f"""
        DROP TABLE IF EXISTS {SCHEMA}.{TABLE_STAGE};
        
        CREATE UNLOGGED TABLE {SCHEMA}.{TABLE_STAGE} (
            uid                 TEXT,
            title               TEXT,
            company_id          TEXT,
            company_name        TEXT,
            company_is_recruiter TEXT,
            company_size        TEXT,
            cantons             TEXT,
            x28_industries      TEXT,
            x28_occupations     TEXT,
            location_raw        TEXT,
            url                 TEXT,
            content_clean       TEXT,
            content_norm        TEXT,
            duplicate_group     TEXT,
            tst_created         TEXT,
            tst_deleted         TEXT,
            source_folder       TEXT,
            source_file         TEXT
        );
    """)
conn.commit()
print("✅ Staging table created")

# Data processing functions
def normalize_text_for_matching(text):
    if text is None:
        return None
    t = unicodedata.normalize("NFD", str(text)).encode("ascii","ignore").decode("ascii")
    t = t.lower()
    t = re.sub(r"\\s+", " ", t).strip()
    return t

def ndjson_stream(path: Path):
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

def first_non_null(d, *keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

def synthesize_uid_from_fields(*parts):
    key = "|".join("" if p is None else str(p) for p in parts)
    return "surr_" + hashlib.sha1(key.encode("utf-8")).hexdigest()

def choose_surrogate_uid(obj):
    company_id   = first_non_null(obj, "company_id")
    company_name = first_non_null(obj, "company_name", "companyName", "company")
    url          = first_non_null(obj, "url")
    title        = first_non_null(obj, "title")
    tst_created  = first_non_null(obj, "tst_created", "created_at", "posted_at", "createdAt")

    combos = [
        (company_id, url, tst_created),
        (company_id, url, title, tst_created),
        (company_name, url, tst_created),
        (company_name, title, url, tst_created),
    ]
    for tup in combos:
        if all(x not in (None, "") for x in tup):
            return synthesize_uid_from_fields(*tup)
    return None

COLS = [
    "uid","title","company_id","company_name","company_is_recruiter","company_size",
    "cantons","x28_industries","x28_occupations","location_raw","url",
    "content_clean","content_norm","duplicate_group","tst_created","tst_deleted",
    "source_folder","source_file",
]

def build_row(obj, source_folder, source_file):
    # Preferred native uid
    native_uid   = first_non_null(obj, "uid", "id")
    title        = first_non_null(obj, "title")
    company_name = first_non_null(obj, "company_name", "companyName", "company")
    company_id   = first_non_null(obj, "company_id")
    url          = first_non_null(obj, "url")
    tst_created  = first_non_null(obj, "tst_created", "created_at", "posted_at", "createdAt")

    uid = str(native_uid) if native_uid else choose_surrogate_uid(obj)
    if not uid:
        return None

    # Require critical downstream fields
    if not (title and company_name and tst_created and url):
        return None

    content_clean = obj.get("content_clean")
    content_norm  = normalize_text_for_matching(content_clean)

    # Keep as text for staging table
    raw_dup = obj.get("duplicate_group")
    dup_str = str(raw_dup) if raw_dup not in (None, "") else None

    return (
        uid, title, company_id, company_name,
        str(obj.get("company_is_recruiter")) if obj.get("company_is_recruiter") is not None else None,
        obj.get("company_size"),
        str(obj.get("cantons")) if obj.get("cantons") is not None else None,
        str(obj.get("x28_industries")) if obj.get("x28_industries") is not None else None,
        str(obj.get("x28_occupations")) if obj.get("x28_occupations") is not None else None,
        obj.get("location_raw"), url, content_clean, content_norm, dup_str,
        str(tst_created) if tst_created else None,
        str(obj.get("tst_deleted")) if obj.get("tst_deleted") else None,
        source_folder, source_file,
    )

print("🚀 Loading data into staging table...")

# Load data into staging table
BASE = Path("/Users/bradyallardice/Desktop/PhD/Projects/KurerAllardice2024/10CompaniesPOCAI/")
FOLDERS = [
    BASE / "240826_panel_data_with_dg",
    BASE / "250826_panel_data_01012024_30062025",
]

def list_json_gz(folder: Path):
    return list(folder.rglob("*.json.gzip"))

files = []
for fld in FOLDERS:
    files.extend(list_json_gz(fld))
files = sorted(set(p.resolve() for p in files))

insert_sql = f"""
INSERT INTO {SCHEMA}.{TABLE_STAGE} ({", ".join(COLS)})
VALUES %s;
"""

def insert_batch(cur, rows):
    if not rows:
        return 0
    execute_values(cur, insert_sql, rows, page_size=5000)
    return len(rows)

total_scanned = total_inserted = 0
for fld in FOLDERS:
    scanned = inserted_here = 0
    these = sorted(set(p.resolve() for p in list_json_gz(fld)))
    
    print(f"📁 Processing folder: {fld.name}")
    
    for i, path in enumerate(these):
        if i % 20 == 0:  # Progress update every 20 files
            print(f"   Processing file {i+1}/{len(these)}: {path.name}")
            
        batch = []
        for obj in ndjson_stream(path):
            scanned += 1
            total_scanned += 1
            row = build_row(obj, source_folder=fld.name, source_file=path.name)
            if row is None:
                continue
            batch.append(row)
            if len(batch) >= 5000:
                with conn.cursor() as cur:
                    cur.execute("SET LOCAL synchronous_commit = off;")
                    inserted_here += insert_batch(cur, batch)
                    conn.commit()
                batch.clear()
        
        if batch:
            with conn.cursor() as cur:
                cur.execute("SET LOCAL synchronous_commit = off;")
                inserted_here += insert_batch(cur, batch)
                conn.commit()
            batch.clear()
    
    total_inserted += inserted_here
    print(f"✅ [{fld.name}] scanned≈{scanned:,} inserted={inserted_here:,}")

print(f"📊 Total: scanned≈{total_scanned:,} inserted={total_inserted:,}")

# Check staging table
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {SCHEMA}.{TABLE_STAGE};")
    count = cur.fetchone()[0]
print(f"✅ Staging table now has {count:,} rows")

🔧 Starting fixed implementation with staging table approach...
🗂️ Creating staging table with all TEXT columns...
✅ Staging table created
🚀 Loading data into staging table...
📁 Processing folder: 240826_panel_data_with_dg
   Processing file 1/187: data_000000000000.json.gzip
   Processing file 21/187: data_000000000020.json.gzip
   Processing file 41/187: data_000000000040.json.gzip
   Processing file 61/187: data_000000000060.json.gzip
   Processing file 81/187: data_000000000080.json.gzip
   Processing file 101/187: data_000000000100.json.gzip
   Processing file 121/187: data_000000000120.json.gzip
   Processing file 141/187: data_000000000140.json.gzip
   Processing file 161/187: data_000000000160.json.gzip
   Processing file 181/187: data_000000000180.json.gzip
✅ [240826_panel_data_with_dg] scanned≈6,006,115 inserted=6,006,115
📁 Processing folder: 250826_panel_data_01012024_30062025
   Processing file 1/4: data_000000000000.json.gzip
✅ [250826_panel_data_01012024_30062025] scanned≈

In [ ]:
# ============================================================================
# MERGE STAGING INTO FINAL TABLE WITH PROPER TYPE CASTING
# ============================================================================

print("🔄 First, let's modify the final table to handle surrogate UIDs...")

# The final table was created with UUID type, but we have surrogate UIDs that are TEXT
# Let's alter the table to use TEXT for uid column
with conn.cursor() as cur:
    # Check if we need to alter the table
    cur.execute("""
        SELECT data_type 
        FROM information_schema.columns 
        WHERE table_name = %s AND column_name = 'uid' AND table_schema = 'public'
    """, [TABLE_FINAL])
    
    current_type = cur.fetchone()[0]
    print(f"Current uid column type: {current_type}")
    
    if current_type == 'uuid':
        print("🔧 Converting uid column from UUID to TEXT to handle surrogate UIDs...")
        # Drop the primary key constraint first
        cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_FINAL} DROP CONSTRAINT job_postings_unified_pkey;")
        # Change column type
        cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_FINAL} ALTER COLUMN uid TYPE TEXT;")
        # Recreate primary key
        cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_FINAL} ADD PRIMARY KEY (uid);")
        print("✅ uid column converted to TEXT")
    else:
        print("✅ uid column already TEXT type")

conn.commit()

print("🔄 Merging staging data into final table...")

# UUID regex pattern to avoid curly brace conflicts with sql.SQL().format()
uuid_regex = r'^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$'

with conn.cursor() as cur:
    cur.execute("SET LOCAL synchronous_commit = off;")
    
    # Use parameterized query to avoid curly brace conflicts with regex
    # Note: removed uid::uuid casting since uid is now TEXT
    cur.execute(sql.SQL("""
        INSERT INTO {}.{} (
            uid, title, company_id, company_name, company_is_recruiter, company_size,
            cantons, x28_industries, x28_occupations, location_raw, url,
            content_clean, duplicate_group, tst_created, tst_deleted,
            source_folder, source_file
        )
        SELECT
            uid,  -- No casting needed since both are TEXT now
            title,
            company_id,
            company_name,
            CASE 
                WHEN company_is_recruiter IS NULL OR company_is_recruiter = '' THEN NULL
                ELSE company_is_recruiter::boolean
            END,
            company_size,
            CASE 
                WHEN cantons IS NULL OR cantons = '' THEN NULL
                WHEN cantons = '[]' OR cantons = 'null' THEN NULL
                ELSE string_to_array(
                    trim(both '[]"'' ' from cantons), 
                    ','
                )
            END AS cantons,
            CASE 
                WHEN x28_industries IS NULL OR x28_industries = '' THEN NULL
                WHEN x28_industries = '[]' OR x28_industries = 'null' THEN NULL
                ELSE string_to_array(
                    trim(both '[]"'' ' from x28_industries), 
                    ','
                )
            END AS x28_industries,
            CASE 
                WHEN x28_occupations IS NULL OR x28_occupations = '' THEN NULL
                WHEN x28_occupations = '[]' OR x28_occupations = 'null' THEN NULL
                ELSE string_to_array(
                    trim(both '[]"'' ' from x28_occupations), 
                    ','
                )
            END AS x28_occupations,
            location_raw,
            url,
            content_clean,
            CASE
                WHEN duplicate_group IS NULL THEN NULL
                WHEN COALESCE(btrim(duplicate_group), '') = '' THEN NULL
                WHEN duplicate_group = 'null' THEN NULL
                WHEN duplicate_group ~* %s THEN duplicate_group::uuid
                ELSE NULL
            END AS duplicate_group,
            CASE
                WHEN tst_created IS NULL OR tst_created = '' THEN NULL
                ELSE tst_created::timestamptz
            END,
            CASE
                WHEN tst_deleted IS NULL OR tst_deleted = '' THEN NULL
                ELSE tst_deleted::timestamptz
            END,
            source_folder,
            source_file
        FROM {}.{}
        ON CONFLICT (uid) DO NOTHING
    """).format(
        sql.Identifier(SCHEMA), sql.Identifier(TABLE_FINAL),
        sql.Identifier(SCHEMA), sql.Identifier(TABLE_STAGE)
    ), [uuid_regex])

conn.commit()
print("✅ Merge into final table complete")

# Check final table count
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {SCHEMA}.{TABLE_FINAL};")
    final_count = cur.fetchone()[0]

print(f"📊 Final table now has {final_count:,} rows")

# Update content_norm using the generated column functionality
print("🔄 Updating content_norm for rows where it's NULL...")
with conn.cursor() as cur:
    cur.execute(f"""
        UPDATE {SCHEMA}.{TABLE_FINAL} 
        SET content_clean = content_clean 
        WHERE content_norm IS NULL AND content_clean IS NOT NULL;
    """)
    cur.execute("ANALYZE public.job_postings_unified;")
    
conn.commit()
print("✅ Content normalization complete and table analyzed")

In [ ]:
# ============================================================================
# QUALITY ASSURANCE CHECKS
# ============================================================================

print("🔍 Running QA checks...")

def qdf(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("📊 Row counts by source:")
result = qdf("""
    SELECT source_folder, COUNT(*) AS n
    FROM public.job_postings_unified
    GROUP BY source_folder ORDER BY source_folder;
""")
print(result)

print("\n📅 Date ranges by source:")
result = qdf("""
    SELECT source_folder,
           MIN(tst_created) AS min_created,
           MAX(tst_created) AS max_created
    FROM public.job_postings_unified
    GROUP BY source_folder ORDER BY source_folder;
""")
print(result)

print("\n🔗 Duplicate group coverage by source:")
result = qdf("""
    SELECT source_folder,
           COUNT(*) AS total,
           COUNT(duplicate_group) AS with_dupgroup,
           ROUND(100.0 * COUNT(duplicate_group)::numeric / NULLIF(COUNT(*),0), 2) AS pct_with_dupgroup
    FROM public.job_postings_unified
    GROUP BY source_folder ORDER BY source_folder;
""")
print(result)

print("\n🎯 Sample 5 rows:")
result = qdf("""
    SELECT uid, company_name, title, tst_created, LEFT(content_clean, 120) AS snippet
    FROM public.job_postings_unified
    ORDER BY tst_created DESC
    LIMIT 5;
""")
print(result)

print("\n🔍 Search smoke test (ILIKE + regex for AI content):")
result = qdf(r"""
    SELECT uid, company_name, title, tst_created
    FROM public.job_postings_unified
    WHERE content_clean ILIKE '%machine learning%'
       OR content_norm ~* '\yai\y'
    ORDER BY tst_created DESC
    LIMIT 10;
""")
print(result)

print("\n📈 Content normalization status:")
result = qdf("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(content_clean) as with_content_clean,
        COUNT(content_norm) as with_content_norm,
        ROUND(100.0 * COUNT(content_norm)::numeric / NULLIF(COUNT(*),0), 2) AS pct_normalized
    FROM public.job_postings_unified;
""")
print(result)

# Optional cleanup
print("\n🧹 Cleaning up staging table...")
with conn.cursor() as cur:
    cur.execute(f"TRUNCATE TABLE {SCHEMA}.{TABLE_STAGE};")
conn.commit()

conn.close()
print("\n✅ QA complete - Connection closed")
print("🎉 Table creation and data loading successfully completed!")

In [ ]:
# ============================================================================
# FIX DATABASE CONNECTION ERROR
# ============================================================================

# The transaction error means the connection is in a failed state
# Let's reset the connection and try again

print("🔧 Fixing database connection...")

try:
    conn.rollback()  # Roll back any failed transaction
    print("✅ Transaction rolled back")
except:
    print("⚠️ No active transaction to rollback")

# Close and reconnect
try:
    conn.close()
    print("✅ Old connection closed")
except:
    print("⚠️ Connection was already closed")

# Reconnect fresh
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd

load_dotenv("config.env")
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)
conn.autocommit = True  # Use autocommit to avoid transaction issues

print("✅ Fresh database connection established")

def qdf(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

# Quick test
try:
    result = qdf("SELECT COUNT(*) as count FROM public.job_postings_unified;")
    print(f"📊 Table has {result['count'].iloc[0]:,} rows")
    print("🎉 Connection is working!")
except Exception as e:
    print(f"❌ Still having issues: {e}")

In [ ]:
# ============================================================================
# CHECK STATUS AND OPTIMIZE MERGE OPERATION
# ============================================================================

print("🔍 Checking current table status...")

# Check staging table count
try:
    result = qdf("SELECT COUNT(*) as count FROM public.job_postings_unified_staging;")
    staging_count = result['count'].iloc[0]
    print(f"📊 Staging table: {staging_count:,} rows")
except Exception as e:
    print(f"❌ Error checking staging table: {e}")

# Check final table count
try:
    result = qdf("SELECT COUNT(*) as count FROM public.job_postings_unified;")
    final_count = result['count'].iloc[0]
    print(f"📊 Final table: {final_count:,} rows")
except Exception as e:
    print(f"❌ Error checking final table: {e}")

# Check if there are any active long-running queries
try:
    result = qdf("""
        SELECT pid, state, query_start, now() - query_start as duration, 
               left(query, 100) as query_snippet
        FROM pg_stat_activity 
        WHERE state = 'active' 
        AND query NOT LIKE '%pg_stat_activity%'
        AND query_start < now() - interval '1 minute'
        ORDER BY query_start;
    """)
    if len(result) > 0:
        print("⚠️ Long-running queries detected:")
        print(result)
        print("\n💡 You may need to cancel these queries if they're stuck")
    else:
        print("✅ No long-running queries detected")
except Exception as e:
    print(f"❌ Error checking active queries: {e}")

print("\n🚀 If the merge is stuck, we should use a batched approach instead...")

In [1]:
# ============================================================================
# OPTIMIZED BATCHED MERGE OPERATION  
# ============================================================================

print("🚀 Starting optimized batched merge operation...")

# First, let's cancel any stuck operations (you might need to restart your kernel if needed)
# Then we'll do a batched merge instead

BATCH_SIZE = 50000  # Process 50k rows at a time
uuid_regex = r'^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$'

# Get total rows to process
try:
    result = qdf("SELECT COUNT(*) as count FROM public.job_postings_unified_staging;")
    total_rows = result['count'].iloc[0]
    print(f"📊 Total rows to process: {total_rows:,}")
    
    if total_rows == 0:
        print("❌ No data in staging table. Please run the data loading step first.")
    else:
        # Process in batches
        batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"📦 Processing in {batches} batches of {BATCH_SIZE:,} rows each")
        
        processed = 0
        
        for batch_num in range(batches):
            offset = batch_num * BATCH_SIZE
            
            print(f"🔄 Processing batch {batch_num + 1}/{batches} (offset {offset:,})...")
            
            # Use a fresh connection for each batch to avoid timeout issues
            conn.autocommit = False
            
            with conn.cursor() as cur:
                cur.execute("SET LOCAL synchronous_commit = off;")
                cur.execute("SET LOCAL work_mem = '256MB';")  # Increase work memory
                
                batch_sql = f"""
                    INSERT INTO public.job_postings_unified (
                        uid, title, company_id, company_name, company_is_recruiter, company_size,
                        cantons, x28_industries, x28_occupations, location_raw, url,
                        content_clean, duplicate_group, tst_created, tst_deleted,
                        source_folder, source_file
                    )
                    SELECT
                        uid,
                        title,
                        company_id,
                        company_name,
                        CASE 
                            WHEN company_is_recruiter IS NULL OR company_is_recruiter = '' THEN NULL
                            ELSE company_is_recruiter::boolean
                        END,
                        company_size,
                        CASE 
                            WHEN cantons IS NULL OR cantons = '' OR cantons = '[]' OR cantons = 'null' THEN NULL
                            ELSE string_to_array(trim(both '[]"\\' ' from cantons), ',')
                        END,
                        CASE 
                            WHEN x28_industries IS NULL OR x28_industries = '' OR x28_industries = '[]' OR x28_industries = 'null' THEN NULL
                            ELSE string_to_array(trim(both '[]"\\' ' from x28_industries), ',')
                        END,
                        CASE 
                            WHEN x28_occupations IS NULL OR x28_occupations = '' OR x28_occupations = '[]' OR x28_occupations = 'null' THEN NULL
                            ELSE string_to_array(trim(both '[]"\\' ' from x28_occupations), ',')
                        END,
                        location_raw,
                        url,
                        content_clean,
                        CASE
                            WHEN duplicate_group IS NULL OR duplicate_group = '' OR duplicate_group = 'null' THEN NULL
                            WHEN duplicate_group ~ %s THEN duplicate_group::uuid
                            ELSE NULL
                        END,
                        CASE
                            WHEN tst_created IS NULL OR tst_created = '' THEN NULL
                            ELSE tst_created::timestamptz
                        END,
                        CASE
                            WHEN tst_deleted IS NULL OR tst_deleted = '' THEN NULL
                            ELSE tst_deleted::timestamptz
                        END,
                        source_folder,
                        source_file
                    FROM (
                        SELECT * FROM public.job_postings_unified_staging
                        ORDER BY uid  -- Consistent ordering
                        LIMIT %s OFFSET %s
                    ) batch
                    ON CONFLICT (uid) DO NOTHING;
                """
                
                cur.execute(batch_sql, [uuid_regex, BATCH_SIZE, offset])
                batch_inserted = cur.rowcount
                processed += batch_inserted
                
                conn.commit()
                
                print(f"    ✅ Batch {batch_num + 1} complete: {batch_inserted:,} rows inserted (total: {processed:,})")
            
            conn.autocommit = True  # Reset to autocommit
        
        print(f"\n🎉 Batched merge complete! Processed {processed:,} rows total")
        
        # Final count check
        result = qdf("SELECT COUNT(*) as count FROM public.job_postings_unified;")
        final_count = result['count'].iloc[0]
        print(f"📊 Final table now has {final_count:,} rows")
        
        # Analyze the table
        print("🔄 Analyzing table...")
        with conn.cursor() as cur:
            cur.execute("ANALYZE public.job_postings_unified;")
        print("✅ Table analyzed")
        
except Exception as e:
    print(f"❌ Error during batched merge: {e}")
    try:
        conn.rollback()
    except:
        pass

🚀 Starting optimized batched merge operation...
❌ Error during batched merge: name 'qdf' is not defined
